In [0]:
schema = "id int, name string, age short, salary double"

data_list = [(100, "Prashant", 45, 45000),
             (101, "Tarun", 36, 33000),
             (102, "David", 48, 28000)]

#sample_df = spark.createDataFrame(data=data_list)
#sample_df = spark.createDataFrame(data=data_list).toDF("id", "name", "age", "salary")
sample_df = spark.createDataFrame(data=data_list, schema=schema)


In [0]:
from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumns({
        "increment": expr("case when salary > 30000 then 3000 else salary * 10/100 end"),
        "revised_salary": expr("salary + increment")
    })
)

salary_df.display()

In [0]:
from pyspark.sql.functions import expr

salary_df = (
    sample_df.withColumn("increment", expr("case when salary > 30000 then 3000 else salary * 10/100 end"))
        .withColumn("salary", expr("salary + increment"))
)

salary_df.display()

In [0]:
import uuid
from pyspark.sql.functions import lit

batch_id = str(uuid.uuid4())

salary_batch_df = sample_df.withColumn("batch_id", lit(batch_id))

salary_batch_df.show()

In [0]:
new_salary_df = (
    salary_df.withColumnsRenamed({
        "increment": "annual_increment",
        "salary": "incremented_salary"
    })
)

new_salary_df.display()

In [0]:
%sql

select * from dev.spark_db.flight_time order by DISTANCE desc

In [0]:
flight_time_df = spark.read.table("dev.spark_db.flight_time")

flight_time_1_df = (
    flight_time_df.selectExpr(
        "fl_date as dep_date",
        "to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date",
        "dep_date + crs_dep_time as crs_dep_time",
        "dep_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time",
        "op_carrier"
    )
)

flight_time_1_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

In [0]:
from pyspark.sql.functions import expr

flight_time_2_df = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", expr("to_date(dep_date + dep_time + wheels_on + taxi_in) as arr_date"))
      .withColumns({
        "crs_dep_time": expr("dep_date + crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": expr("arr_date + arr_time"),
      })
)

flight_time_2_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'").display()

In [0]:
from pyspark.sql.functions import to_date, col

flight_time_2_df = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      .withColumn("arr_date", to_date(col("dep_date") + col("dep_time") + col("wheels_on") + col("taxi_in")))
      .withColumns({
        "crs_dep_time": col("dep_date") + col("crs_dep_time"),
        "dep_time": col("dep_date") + col("dep_time"),
        "crs_arr_time": col("arr_date") + col("crs_arr_time"),
        "arr_time": col("arr_date") + col("arr_time"),
      })
)

flight_time_2_df.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).display()